# Training Config Walkthrough

This notebook explains the YAML paramters that are used to configure model training (inputs/outputs, training hyperparameters, and optional compression like quantization/pruning).

**Config file:** `config_training.yml`


In [ ]:
# Display the current yaml file 
import yaml
from pathlib import Path
from pprint import pprint

REPO = Path.cwd().resolve()
cfg_path = REPO / "Driver" / "CartPoleSimulation" / "SI_Toolkit_ASF" / "config_training.yml"

with open(cfg_path, "r") as f:
    cfg = yaml.safe_load(f)

pprint(cfg)

## 1) Top level switches

### `library`
- Chooses training backend.
- Typical values: `TF` or `Pytorch`.

### `modeling/NET_NAME`
- Names the network architecture recipe to build.
- Example here: `GRU-64H1-64H2`


## 2) Paths: Where data comes from and where artifacts go

### `paths/path_to_experiment`
- This is the experiment folder name (relative to `PATH_TO_EXPERIMENT_FOLDERS`).
- This is the workspace containing:
  - `Recordings/Train`, `Recordings/Validate`, `Recordings/Test`
  - `NormalizationInfo/`
  - `Models/`

### `paths/PATH_TO_EXPERIMENT_FOLDERS`
- Base directory that contains experiment folders.

### `paths/DATA_FOLDER`
- Subfolder inside the experiment folder that holds recordings (usually called `Recordings`).

If the wrong experiment path is provided the training can't find CSVs or normalization info.


In [ ]:
from pathlib import Path

exp_root = Path(cfg['paths']['PATH_TO_EXPERIMENT_FOLDERS'])
exp = exp_root / cfg['paths']['path_to_experiment']
print('Experiment folder:', exp.resolve())
print('Has Recordings?:', (exp/cfg['paths']['DATA_FOLDER']).exists())
print('Train folder:', (exp/cfg['paths']['DATA_FOLDER']/'Train'))
print('Validate folder:', (exp/cfg['paths']['DATA_FOLDER']/'Validate'))
print('Test folder:', (exp/cfg['paths']['DATA_FOLDER']/'Test'))

## 3) Define the training problem

All items below live under `training_default`.

### Inputs
- `state_inputs`: columns are read from the CSV and fed into the network.
- `control_inputs`, `setpoint_inputs`, `translation_invariant_variables`: optional extra groups depending on pipeline.

### Outputs
- `outputs`: columns the network learns to predict.


In [ ]:
inputs = cfg['training_default']['state_inputs']
outputs = cfg['training_default']['outputs']
print('Inputs (state_inputs):')
for x in inputs: print(' -', x)
print('\nOutputs:')
for y in outputs: print(' -', y)

## 4) Core training parameters

### `EPOCHS`
- Number of full passes over the training dataset.
- More epochs can improve fit, but increases overfitting risk.

### `BATCH_SIZE`
- Number of samples used in each gradient update.

### `SEED`
- Used to reproduce data shuffling and parameter init.

### Learning-rate block `LR`
- `INITIAL`: starting learning rate.
- `REDUCE_LR_ON_PLATEAU`: if `True`, reduce LR when validation loss stops improving.
- `PATIENCE`: The number of epochs without improvement that we tolerate.
- `DECREASE_FACTOR`: multiplier (e.g. 0.5 halves LR).
- `MINIMAL`: floor for LR.
- `MIN_DELTA`: how much improvement counts.


In [ ]:
td = cfg['training_default']
print('EPOCHS:', td['EPOCHS'])
print('BATCH_SIZE:', td['BATCH_SIZE'])
print('SEED:', td['SEED'])
print('LR:', td['LR'])

## 5) Sequence handling

### `WASH_OUT_LEN`
- Number of initial timesteps ignored for loss.
- Purpose: let the state “warm up” so the model isn’t penalized for its initial state.

### `POST_WASH_OUT_LEN`
- Additional tail length after washout used for loss

### `SHIFT_LABELS`
- Shifts outputs relative to inputs.
- `0` means predict current output from current input window.
- `+1` means predict next step output.

## 6) Data normalization and augmentation

### `NORMALIZE`
- If `True`, training uses normalization info (mean/std or affine scaling) computed for the experiment.
- This usually stabilizes training and makes tuning the learning rate easier.

### `ON_FLY_DATA_GENERATION`
- If `True`, training can generate/augment sequences dynamically rather than only using pre-saved CSV segments.

### `CONFIG_SERIES_MODIFICATION`
- A macro that can modify the series before training.
- Example mode: `train_for_random_vertical_angle_shift`
- `NOISE_LEVEL/FEATURES`: additive noise is injected into the input features.


In [ ]:
cfg.get('CONFIG_SERIES_MODIFICATION', {})

## 7) Loss function

### `LOSS_MODE`
- `squared` → mean squared error style loss.
- Other modes may exist in the toolkit (e.g. absolute) depending on backend.

## 8) Model construction flags

### `CONSTRUCT_NETWORK`
- Controls how the architecture string is interpreted.

### `PLOT_WEIGHTS_DISTRIBUTION`
- If `True`, stores diagnostics plots of learned weights.

### `VALIDATE_ALSO_ON_TRAINING_SET`
- If `True`, computes validation metrics on training data too.


## 9) Optional regularization

### `REGULARIZATION/ACTIVATED`
If enabled, adds L1/L2 penalties.

- `KERNEL`: weights (main matrices)
- `BIAS`: bias vectors
- `ACTIVITY`: penalizes activations


In [ ]:
cfg.get('REGULARIZATION', {})

## 10) Optional quantization (for deployment / hls4ml)

### `QUANTIZATION/ACTIVATED`
- Enables quantization aware training or post training quantization hooks (depends on backend).

Important fields:
- `QUANTIZATION_DATASET`: target fixed point type (e.g. `ap_fixed<12,2>`)
- `ACTIVATION/bits`: activation bitwidth
- `KERNEL` / `BIAS` / `RECURRENT`: bitwidths, integer bits, symmetry

**Effect:**
- Reduces model size to enable FPGA friendly inferencing.
- Usually increases error slightly, but this trade off depends on bitwidth.

In [ ]:
cfg.get('QUANTIZATION', {})

## 11) Optional pruning (sparsity)

### `PRUNING/ACTIVATED`
- Enables sparse training.

Two schedule options shown:
- `CONSTANT_SPARSITY`: jump to a target sparsity and keep it.
- `POLYNOMIAL_DECAY`: gradually increase sparsity over time.

Key knobs:
- `target_sparsity`: fraction of weights set to zero.
- `begin_step_in_epochs`: when pruning starts.
- `end_step_in_training_fraction`: fraction of training when pruning ends.
- `frequency_per_epoch`: how often masks update.

Note:
- Can reduce model size/compute.
- Over aggressive pruning early can prevent learning.


In [ ]:
cfg.get('PRUNING', {})